In [3]:
!pip uninstall -y langchain-community

Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2


In [4]:
import os
import json
import numpy as np
import faiss

from google import genai

print("Libraries imported successfully!")

Libraries imported successfully!


In [6]:
from google import genai
import os

print("Gemini imported successfully!")

Gemini imported successfully!


In [7]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in Colab Secrets")

print("API key loaded successfully!")

API key loaded successfully!


In [8]:
client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client connected successfully!")

Gemini client connected successfully!


In [9]:
import os

print(os.listdir("/content"))

['.config', '.ipynb_checkpoints', 'career_notes.zip', 'sample_data']


In [10]:
import zipfile
import os

zip_path = "/content/career_notes.zip"
notes_folder = "/content/career_notes"

os.makedirs(notes_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(notes_folder)

print("Career notes extracted successfully!")
print(os.listdir(notes_folder))

Career notes extracted successfully!
['career_guide.txt']


In [12]:
from pathlib import Path

notes_path = Path("/content/career_notes")

print("Career notes folder:", notes_path)
print("Contents:", list(notes_path.iterdir()))

Career notes folder: /content/career_notes
Contents: [PosixPath('/content/career_notes/career_guide.txt')]


In [13]:
for item in notes_path.iterdir():
    print(item, "->", "FILE" if item.is_file() else "FOLDER")

/content/career_notes/career_guide.txt -> FOLDER


In [14]:
career_folder = notes_path / "career_guide.txt"

print("Inside career_guide.txt:")
print(list(career_folder.iterdir()))

Inside career_guide.txt:
[PosixPath('/content/career_notes/career_guide.txt/genai_engineer.txt'), PosixPath('/content/career_notes/career_guide.txt/ml_engineer.txt'), PosixPath('/content/career_notes/career_guide.txt/data_analyst.txt'), PosixPath('/content/career_notes/career_guide.txt/business_analyst.txt'), PosixPath('/content/career_notes/career_guide.txt/resume_interview_roadmap.txt')]


In [15]:
career_files = list(career_folder.glob("*.txt"))

for file in career_files:
    print("Reading:", file)
    text = file.read_text(encoding="utf-8", errors="ignore")
    print(text[:1000])

Reading: /content/career_notes/career_guide.txt/genai_engineer.txt
ROLE GUIDE: GENERATIVE AI / LLM ENGINEER (ENTRY-LEVEL)

Overview:
This is an emerging role focused on building applications powered by large language models (LLMs) — chatbots, RAG systems, AI agents, and content generation tools. It's less about training models from scratch and more about integrating and orchestrating existing models effectively.

Core Skills Required:
- Prompt engineering: zero-shot, few-shot, chain-of-thought techniques
- Working with LLM APIs (OpenAI, Anthropic Claude, Google Gemini)
- Embeddings and vector databases (FAISS, Pinecone, Chroma) for semantic search
- RAG (Retrieval-Augmented Generation) pipeline design
- Orchestration frameworks: LangChain or LlamaIndex
- Basic app deployment: Streamlit, FastAPI, or similar for building demos
- Understanding of guardrails and responsible AI practices (input validation, hallucination checks)

How to Switch Into This Role:
1. Build at least one deployed R

In [16]:
all_notes = []

for file in career_files:
    text = file.read_text(encoding="utf-8", errors="ignore")

    if text.strip():
        all_notes.append(text)

print("Number of notes loaded:", len(all_notes))

Number of notes loaded: 5


In [17]:
chunks = []

for text in all_notes:
    words = text.split()

    chunk_size = 150

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])

        if chunk.strip():
            chunks.append(chunk)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Number of chunks: 11

--- Chunk 1 ---
ROLE GUIDE: GENERATIVE AI / LLM ENGINEER (ENTRY-LEVEL) Overview: This is an emerging role focused on building applications powered by large language models (LLMs) — chatbots, RAG systems, AI agents, and content generation tools. It's less about training models from scratch and more about integrating and orchestrating existing models effectively. Core Skills Required: - Prompt engineering: zero-shot, few-shot, chain-of-thought techniques - Working with LLM APIs (OpenAI, Anthropic Claude, Google Gemini) - Embeddings and vector databases (FAISS, Pinecone, Chroma) for semantic search - RAG (Retrieval-Augmented Generation) pipeline design - Orchestration frameworks: LangChain or LlamaIndex - Basic app deployment: Streamlit, FastAPI, or similar for building demos - Understanding of guardrails and responsible AI practices (input validation, hallucination checks) How to Switch Into This Role: 1. Build at least one deployed RAG project (e.g. a PDF Q&A bot o

In [18]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client connected successfully!")

Gemini client connected successfully!


In [19]:
response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=chunks[0]
)

embedding = response.embeddings[0].values

print("Embedding created successfully!")
print("Embedding length:", len(embedding))

Embedding created successfully!
Embedding length: 3072


In [20]:
!pip install -q faiss-cpu

In [21]:
import faiss
import numpy as np

print("FAISS and NumPy imported successfully!")

FAISS and NumPy imported successfully!


In [22]:
chunk_embeddings = []

for i, chunk in enumerate(chunks):
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk
    )

    vector = response.embeddings[0].values
    chunk_embeddings.append(vector)

print("Embeddings created:", len(chunk_embeddings))

Embeddings created: 11


In [23]:
embedding_matrix = np.array(
    chunk_embeddings,
    dtype="float32"
)

print("Embedding matrix shape:", embedding_matrix.shape)

Embedding matrix shape: (11, 3072)


In [24]:
dimension = embedding_matrix.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embedding_matrix)

print("FAISS index created successfully!")
print("Number of vectors:", index.ntotal)

FAISS index created successfully!
Number of vectors: 11


In [25]:
question = "How can I become a data analyst?"

response = client.models.embed_content(
    model="gemini-embedding-001",
    contents=question
)

question_embedding = np.array(
    [response.embeddings[0].values],
    dtype="float32"
)

distances, indices = index.search(
    question_embedding,
    k=min(3, len(chunks))
)

print("Retrieved chunks:")

for rank, idx in enumerate(indices[0]):
    print(f"\n--- Result {rank + 1} ---")
    print(chunks[idx])
    print("Distance:", distances[0][rank])

Retrieved chunks:

--- Result 1 ---
ROLE GUIDE: DATA ANALYST Overview: A Data Analyst collects, cleans, and interprets data to help organizations make better decisions. Entry-level roles focus heavily on reporting and dashboarding, while senior roles move toward predictive analysis and strategy. Core Skills Required: - SQL (joins, aggregations, window functions) — the single most important skill for this role - Excel (pivot tables, VLOOKUP/XLOOKUP, basic formulas) - Python or R for data manipulation (Pandas is the most common library) - Data visualization tools: Power BI or Tableau - Statistics fundamentals: mean, median, standard deviation, correlation, hypothesis testing - Strong communication skills to present findings to non-technical stakeholders How to Switch Into This Role: 1. Learn SQL first — it is used in nearly every data analyst job posting. 2. Build 2-3 portfolio projects using public datasets (Kaggle is a good source) and publish dashboards on Power BI or Tableau Public. 

In [26]:
MENTOR_PROMPT = """
You are SmartHire AI Career Mentor.

Answer the user's question using ONLY the career notes provided below.

If the answer cannot be found in the career notes, say:
"I don't know based on the provided career notes."

Do not invent information.
Do not use outside knowledge.

Career notes:
{context}

User question:
{question}

Answer clearly and practically.
"""

In [27]:
def ask_mentor_rag(question, top_k=3):

    # Embed the question
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    )

    question_embedding = np.array(
        [response.embeddings[0].values],
        dtype="float32"
    )

    # Retrieve relevant chunks
    distances, indices = index.search(
        question_embedding,
        k=min(top_k, len(chunks))
    )

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    context = "\n\n".join(retrieved_chunks)

    # Create prompt
    prompt = MENTOR_PROMPT.format(
        context=context,
        question=question
    )

    # Generate answer
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [29]:
def ask_mentor_rag(question, top_k=3):

    # 1. Embed the question
    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    )

    question_embedding = np.array(
        [response.embeddings[0].values],
        dtype="float32"
    )

    # 2. Retrieve relevant chunks from FAISS
    distances, indices = index.search(
        question_embedding,
        k=min(top_k, len(chunks))
    )

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    # 3. Combine retrieved chunks
    context = "\n\n".join(retrieved_chunks)

    # 4. Create the RAG prompt
    prompt = MENTOR_PROMPT.format(
        context=context,
        question=question
    )

    # 5. Generate the answer
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

In [30]:
question = "How can I become a Data Analyst?"

answer = ask_mentor_rag(question)

print(answer)

Based on the provided career notes, here is how you can switch into a Data Analyst role:

1. **Learn SQL First:** Start here, as SQL is the single most important skill and is used in nearly every data analyst job posting.
2. **Build Portfolio Projects:** Create 2–3 portfolio projects using public datasets (Kaggle is a good source) and publish your dashboards on Power BI or Tableau Public. Ensure your projects do not just clean data, but also draw conclusions and recommendations.
3. **Practice Plain-Language Communication:** Practice explaining your findings in plain language rather than using technical jargon, as presenting findings to non-technical stakeholders is a key requirement.
4. **Apply for Junior Roles or Internships:** Apply for internships or junior analyst roles even if you do not have a data-specific degree (many analysts come from finance, business, or engineering backgrounds).
5. **Develop Additional Core Skills:** Build proficiency in Excel (pivot tables, VLOOKUP/XLOOKU

In [31]:
question = "Who is the Prime Minister of India?"

answer = ask_mentor_rag(question)

print(answer)

I don't know based on the provided career notes.


In [32]:
question = "What skills are useful for both a Data Analyst and Business Analyst?"

answer = ask_mentor_rag(question)

print(answer)

Based on the provided career notes, the skills useful for both a Data Analyst and a Business Analyst are:

* **SQL:** Crucial for Data Analysts (used for joins, aggregations, and window functions) and necessary at a basic level for Business Analysts (used for data checks and validating data).
* **Excel:** Listed as a core skill for both roles to work with data.
* **Communication Skills:** Both roles require strong verbal and written communication skills to present findings, problems, and recommendations to stakeholders.
